# I. NEURAL NETWORK TRAINING

## 1. CONFIG

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import matplotlib.pyplot as plt

DATA_PATH = "data/preprocessed_features.csv"
MODEL_DIR = "model/nn_per_target_models"
RESULTS_PATH = "results/nn_per_target_results.csv"

BATCH_SIZE = 256
N_EPOCHS = 100
LEARNING_RATE = 5e-3
RANDOM_STATE = 42

EARLY_STOPPING_PATIENCE = 10
MIN_DELTA = 1e-4

os.makedirs(MODEL_DIR, exist_ok=True)

## 2. LOAD DATA

In [ ]:
df = pd.read_csv(DATA_PATH)

user_col = "pass__user_id"
output_prefixes = (
    "num__liwc_",
    "num__topic_",
    "num__genius_",
    "num__music_track_",
)

output_cols = [c for c in df.columns if c.startswith(output_prefixes)]
input_cols = [c for c in df.columns if c not in output_cols]

input_cols.remove(user_col)

print(f"Inputs: {len(input_cols)}")
print(f"Targets: {len(output_cols)}")

## 3. TRAIN / TEST SPLIT WITH USER BLOCKING & SCALE FEATURES

In [ ]:

groups = df[user_col].values
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.1,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(gss.split(df, groups=groups))

X_train = df.loc[train_idx, input_cols]
X_test  = df.loc[test_idx, input_cols]

y_train = df.loc[train_idx, output_cols]
y_test  = df.loc[test_idx, output_cols]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test_scaled, dtype=torch.float32)

## 4. MODEL DEFINITION

In [ ]:
class SingleOutputNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(256, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

## 5. TRAINING FUNCTION

In [ ]:
def train_single_target(
    target_name,
    X_train_tensor,
    X_test_tensor,
    y_train,
    y_test
):
    y_train_tensor = torch.tensor(
        y_train[target_name].values,
        dtype=torch.float32
    )
    y_test_tensor = torch.tensor(
        y_test[target_name].values,
        dtype=torch.float32
    )

    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    test_loader = DataLoader(
        TensorDataset(X_test_tensor, y_test_tensor),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    model = SingleOutputNN(input_dim=X_train_tensor.shape[1])
    criterion = nn.SmoothL1Loss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3,
        verbose=True
    )

    train_losses, test_losses = [], []

    best_test_loss = float("inf")
    best_state_dict = None
    patience_counter = 0

    for epoch in range(N_EPOCHS):
        # TRAIN
        model.train()
        running_train_loss = 0.0

        for Xb, yb in train_loader:
            optimizer.zero_grad()
            preds = model(Xb)
            loss = criterion(preds, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running_train_loss += loss.item() * Xb.size(0)

        epoch_train_loss = running_train_loss / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)

        # TEST
        model.eval()
        running_test_loss = 0.0

        with torch.no_grad():
            for Xb, yb in test_loader:
                preds = model(Xb)
                loss = criterion(preds, yb)
                running_test_loss += loss.item() * Xb.size(0)

        epoch_test_loss = running_test_loss / len(test_loader.dataset)
        test_losses.append(epoch_test_loss)

        print(
            f"Epoch [{epoch+1}/{N_EPOCHS}] "
            f"- Train Loss: {epoch_train_loss:.4f} "
            f"- Test Loss: {epoch_test_loss:.4f}"
        )

        scheduler.step(epoch_test_loss)

        # EARLY STOPPING
        if epoch_test_loss < best_test_loss - MIN_DELTA:
            best_test_loss = epoch_test_loss
            best_state_dict = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(
                    f"Early stopping at epoch {epoch+1} "
                    f"(best test loss: {best_test_loss:.4f})"
                )
                break

    # Restore best model
    model.load_state_dict(best_state_dict)

    # Plot loss curves
    plt.figure(figsize=(6,4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(test_losses, label="Test Loss")
    plt.title(f"Loss Curve - {target_name}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    return model, best_test_loss


## 6. TRAIN ONE MODEL PER TARGET

In [ ]:
results = []

for i, target in enumerate(output_cols, 1):
    print(f"\n[{i}/{len(output_cols)}] Training: {target}")

    model, mse = train_single_target(
        target,
        X_train_tensor,
        X_test_tensor,
        y_train,
        y_test
    )

    model_path = os.path.join(MODEL_DIR, f"nn_{target}.pth")
    torch.save(model.state_dict(), model_path)

    results.append({
        "target": target,
        "mse": mse
    })

# SAVE RESULTS
results_df = pd.DataFrame(results).sort_values("mse")
results_df.to_csv(RESULTS_PATH, index=False)

print("\n=== BEST PREDICTED TARGETS ===")
print(results_df.head(10))

print("\n=== WORST PREDICTED TARGETS ===")
print(results_df.tail(10))

print(f"\nModels saved to: {MODEL_DIR}")
print(f"Results saved to: {RESULTS_PATH}")


# II. PER-TARGET NEURAL NETWORK INTERPRETATION

## 1. CONFIG

In [ ]:
DATA_PATH = "data/preprocessed_features.csv"
MODEL_DIR = "model/nn_per_target_models"
OUTPUT_DIR = "results/nn_interpretation"

N_BASELINE_SAMPLES = 1000     # for importance
N_PDP_POINTS = 20             # per feature
TOP_K_FEATURES = 15           # per target

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. LOAD DATA & SCALE INPUTS

In [ ]:
df = pd.read_csv(DATA_PATH)

user_col = "pass__user_id"
output_prefixes = (
    "num__liwc_",
    "num__topic_",
    "num__genius_",
    "num__music_track_",
)

output_cols = [c for c in df.columns if c.startswith(output_prefixes)]
input_cols = [c for c in df.columns if c not in output_cols]
input_cols.remove(user_col)

X = df[input_cols].values
feature_names = input_cols

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

# Subsample for speed
idx = np.random.choice(len(X_tensor), size=min(N_BASELINE_SAMPLES, len(X_tensor)), replace=False)
X_base = X_tensor[idx]

## 3. INTERPRET EACH MODEL

In [ ]:
for target in output_cols:

    print(f"\nInterpreting {target}")

    # Load model
    model = SingleOutputNN(input_dim=X_tensor.shape[1])
    model.load_state_dict(
        torch.load(os.path.join(MODEL_DIR, f"nn_{target}.pth"))
    )
    model.eval()

    with torch.no_grad():
        baseline_preds = model(X_base)

    # --------------------------------------------------------
    # 1. PERMUTATION FEATURE IMPORTANCE
    # --------------------------------------------------------

    importances = []

    for j, fname in enumerate(feature_names):
        X_perm = X_base.clone()
        X_perm[:, j] = X_perm[torch.randperm(len(X_perm)), j]

        with torch.no_grad():
            perm_preds = model(X_perm)

        importance = torch.mean(
            torch.abs(perm_preds - baseline_preds)
        ).item()

        importances.append((fname, importance))

    importance_df = (
        pd.DataFrame(importances, columns=["feature", "importance"])
        .sort_values("importance", ascending=False)
        .head(TOP_K_FEATURES)
    )

    importance_path = os.path.join(
        OUTPUT_DIR,
        f"importance_{target}.csv"
    )
    importance_df.to_csv(importance_path, index=False)

    # --------------------------------------------------------
    # 2. PARTIAL DEPENDENCE (TOP FEATURES ONLY)
    # --------------------------------------------------------

    pdp_rows = []

    for feature in importance_df["feature"]:
        j = feature_names.index(feature)

        values = torch.linspace(
            X_base[:, j].quantile(0.05),
            X_base[:, j].quantile(0.95),
            N_PDP_POINTS
        )

        for v in values:
            X_mod = X_base.clone()
            X_mod[:, j] = v

            with torch.no_grad():
                y_hat = model(X_mod).mean().item()

            pdp_rows.append({
                "feature": feature,
                "feature_value": v.item(),
                "predicted_target": y_hat
            })

    pdp_df = pd.DataFrame(pdp_rows)

    pdp_path = os.path.join(
        OUTPUT_DIR,
        f"pdp_{target}.csv"
    )
    pdp_df.to_csv(pdp_path, index=False)

print("\nInterpretation complete.")
print(f"Results saved to: {OUTPUT_DIR}")


# III. GLOBAL INTERPRETATION OVERVIEW

## 1. CONFIG & HELPERS

In [ ]:
INTERP_DIR = "results/nn_interpretation"
OUTPUT_SUMMARY = "results/nn_model_interpretation_overview.csv"
OUTPUT_TEXT = "results/nn_model_interpretation_summary.txt"

MIN_EFFECT_SIZE = 0.05    # ignore tiny slopes
TOP_K = 5                # features per target

def estimate_slope(pdp_df):
    """
    Estimate linear slope of feature -> prediction
    """
    x = pdp_df["feature_value"].values
    y = pdp_df["predicted_target"].values

    if len(np.unique(x)) < 2:
        return 0.0

    slope = np.polyfit(x, y, 1)[0]
    return slope

## 2. SUMMARIZE TARGETS

In [ ]:
rows = []

targets = sorted(
    set(
        f.replace("importance_", "").replace(".csv", "")
        for f in os.listdir(INTERP_DIR)
        if f.startswith("importance_")
    )
)

for target in targets:
    print(f"Summarizing {target}")

    imp_path = os.path.join(INTERP_DIR, f"importance_{target}.csv")
    pdp_path = os.path.join(INTERP_DIR, f"pdp_{target}.csv")
    if not os.path.exists(pdp_path):
        continue

    importance_df = pd.read_csv(imp_path).head(TOP_K)
    pdp_df = pd.read_csv(pdp_path)

    for _, row in importance_df.iterrows():
        feature = row["feature"]
        importance = row["importance"]

        feature_pdp = pdp_df[pdp_df["feature"] == feature]
        slope = estimate_slope(feature_pdp)

        if abs(slope) < MIN_EFFECT_SIZE:
            continue  # skip flat features
        direction = "positive" if slope > 0 else "negative"

        rows.append({
            "target": target,
            "feature": feature,
            "importance": importance,
            "effect_direction": direction,
            "effect_slope": slope
        })

## 3. CREATE OVERVIEW TABLE

In [ ]:
overview_df = pd.DataFrame(rows).sort_values(["target", "importance"], ascending=[True, False])
overview_df.to_csv(OUTPUT_SUMMARY, index=False)
print("\nNN interpretation overview created:", OUTPUT_SUMMARY)

# HUMAN-READABLE SUMMARY
summary_rows = []
for target in overview_df["target"].unique():
    subset = overview_df[overview_df["target"] == target]
    pos = subset[subset["effect_direction"] == "positive"]["feature"].tolist()
    neg = subset[subset["effect_direction"] == "negative"]["feature"].tolist()

    statement = []
    if pos:
        statement.append(f"↑ {', '.join(pos[:3])}")
    if neg:
        statement.append(f"↓ {', '.join(neg[:3])}")

    summary_rows.append({"target": target, "summary": " | ".join(statement)})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_TEXT, index=False, header=False)
print(f"Readable summary saved to: {OUTPUT_TEXT}")